# A walk through ztra

This notebook runs the whole loop on the example lab in `examples/world`: look at the bench, check a protocol, watch it run step by step, see what a good error looks like, and compare a (pretend) real run against the prediction.

Everything here uses ztra as a plain Python library — the same code the `ztra` CLI wraps.

## The bench

A world is three YAML files. Evaluating it in a cell draws it: the deck, each plate, the tip racks, and the vials with their fill levels (❄ marks a frozen vial). Hover a well for its contents.

In [1]:
from pathlib import Path
from ztra.world import World

world = World.load(Path("world"))
world

World(inventory=Inventory(version=1, reagents={'water': Reagent(hazard=<Hazard.inert: 'inert'>, concentration=None, msds=None, density_mg_per_ul=1.0), 'hcl_1m': Reagent(hazard=<Hazard.acid: 'acid'>, concentration='1 M', msds='msds/hcl.pdf', density_mg_per_ul=1.0), 'naoh_1m': Reagent(hazard=<Hazard.base: 'base'>, concentration='1 M', msds='msds/naoh.pdf', density_mg_per_ul=1.0), 'enzyme_x': Reagent(hazard=<Hazard.inert: 'inert'>, concentration='10 U/uL', msds=None, density_mg_per_ul=1.0)}, vials={'V_water': Vial(reagent='water', volume_ul=1000.0, state=<ThermalState.thawed: 'thawed'>, freeze_thaw_cycles=0, consumed=False), 'V_hcl': Vial(reagent='hcl_1m', volume_ul=200.0, state=<ThermalState.thawed: 'thawed'>, freeze_thaw_cycles=0, consumed=False), 'V_naoh': Vial(reagent='naoh_1m', volume_ul=200.0, state=<ThermalState.thawed: 'thawed'>, freeze_thaw_cycles=0, consumed=False), 'V_enzyme': Vial(reagent='enzyme_x', volume_ul=150.0, state=<ThermalState.frozen: 'frozen'>, freeze_thaw_cycles=1, consumed=False)}, plates={'P1': Plate(labware='corning_96_wellplate_360ul_flat', wells={'A1': [Liquid(reagent='water', volume_ul=50.0)]})}), deck=Deck(version=1, slots={'1': Slot(entity='P1', trash=False), '2': Slot(entity='TR1', trash=False), '3': Slot(entity='TIPS1', trash=False), '12': Slot(entity=None, trash=True)}, tube_racks={'TR1': TubeRack(labware='opentrons_24_tuberack_nest_1.5ml_snapcap')}, tip_racks={'TIPS1': TipRack(labware='opentrons_96_tiprack_300ul', used=['A1', 'B1'])}, linker={'V_water': Link(rack='TR1', well='A1'), 'V_hcl': Link(rack='TR1', well='A2'), 'V_naoh': Link(rack='TR1', well='A3'), 'V_enzyme': Link(rack='TR1', well='B1')}), hardware=Hardware(version=1, robot=Robot(vendor='opentrons', model=<RobotModel.ot2: 'ot2'>, api_level='2.16'), pipettes=[Pipette(name='p300_single_gen2', mount=<Mount.right: 'right'>, channels=1, min_ul=20.0, max_ul=300.0, tip_labware=['opentrons_96_tiprack_300ul'], accuracy=Accuracy(systematic_pct=2.0, random_pct=1.0, random_ul=0.5))], labware={'corning_96_wellplate_360ul_flat': LabwareDef(kind=<LabwareKind.plate: 'plate'>, rows=8, cols=12, well_max_ul=360.0, tip_volume_ul=None, height_mm=14.2), 'opentrons_24_tuberack_nest_1.5ml_snapcap': LabwareDef(kind=<LabwareKind.tube_rack: 'tube_rack'>, rows=4, cols=6, well_max_ul=1500.0, tip_volume_ul=None, height_mm=43.0), 'opentrons_96_tiprack_300ul': LabwareDef(kind=<LabwareKind.tip_rack: 'tip_rack'>, rows=8, cols=12, well_max_ul=None, tip_volume_ul=300.0, height_mm=64.7)}, sensors={'scale_1': Sensor(kind=<SensorKind.plate_mass: 'plate_mass'>, observes=Observes(entity='P1', wells=[], columns=[]), sigma=0.5, unit='mg', read_time_s=5.0), 'camera_1': Sensor(kind=<SensorKind.well_volume: 'well_volume'>, observes=Observes(entity='P1', wells=[], columns=[1]), sigma=5.0, unit='uL', read_time_s=2.0)}, safe_envelope=SafeEnvelope(temperature_c=Range(min=4.0, max=40.0), max_flow_rate_ul_s=300.0)))

## Checking a protocol

The protocol dilutes an enzyme 1:10 into five wells. We compile it with an observation budget: a scale reading every three transfers, so a failed run can be localized later.

In [2]:
from ztra.protocol import Protocol
from ztra.compiler import compile
from ztra.schedule import Budget

protocol = Protocol.load(Path("protocols/enzyme_dilution.yaml"))
budget = Budget.parse("sensor=scale_1,every=3")
result = compile(world, protocol, budget=budget)
print(len(result.pir), "checked steps,", len(result.outcomes), "predicted outcome(s)")
result.outcomes[0].cost.to_dict()

22 checked steps, 1 predicted outcome(s)


{'thaws': 1,
 'transfers': 10,
 'aspirations': 10,
 'mixes': 5,
 'tips_used': 15,
 'observations': 6,
 'reagent_consumed_ul': {'enzyme_x': 100.0, 'water': 900.0},
 'estimated_time_s': 240.0}

## Watching it run

A trace replays the lowered program one robot step at a time with ideal pipettes. Drag the slider (or press play) to watch the wells fill, the vials drain, and the tips get used up. The final frame is exactly the world the compiler predicted.

In [3]:
from ztra.viz import trace, animate_html
from IPython.display import HTML

frames = trace(world, protocol, budget=budget)
HTML(animate_html(frames, title=protocol.name))

## When it can't work

The same compiler refuses a protocol the lab cannot satisfy — before anything moves. This one loops until a vial runs dry; the error names the step, the vial, and what to do.

In [4]:
from ztra.compiler_errors import CompileError

bad = Protocol.load(Path("protocols/bad_loop_drains_vial.yaml"))
try:
    compile(world, bad)
except CompileError as e:
    err = e.to_dict()
{k: err[k] for k in ("code", "physical_law", "resource", "expected", "actual", "hint") if k in err}

{'code': 'E_VOLUME',
 'physical_law': 'cannot aspirate more than is present',
 'resource': 'V_hcl',
 'expected': '>= 60 uL',
 'actual': '20 uL',
 'hint': 'reduce the volume or add another source vial'}

## Prediction vs reality

For the branching demo protocol, the simulator predicts what the scale should read (with the pipettes' real-world sloppiness folded in), and the diff engine compares that against telemetry from a run. Here the recorded run came up 8 mg light, so the branch took its `otherwise` arm and the deviation is classified.

In [5]:
from ztra.simulate import simulate, Noise
from ztra.sensors import Telemetry
from ztra.diff import diff
from ztra.store import EXPECTED_SEEDS

demo = Protocol.load(Path("protocols/demo.yaml"))
compiled = compile(world, demo)
sim = simulate(world, compiled.pir, Noise.normal(), seeds=EXPECTED_SEEDS)
telemetry = Telemetry.load(Path("telemetry/demo_short_fill.yaml"))
report, observed = diff(compiled, sim, telemetry, None)
report

reading,entity,metric,predicted,observed,delta,sigma,verdict
after_fill,P1,mass_mg,220,211.6,-8.4,4.01456,● VERIFIED_WITHIN_SENSOR_NOISE


## Where to go from here

The [User Guide](../docs/USER_GUIDE.md) covers every command, [WORLD_MODEL.md](../docs/WORLD_MODEL.md) every field of the world files, and `ztra init` scaffolds a project of your own. The store, the runtime and the MCP server (so an agent can drive this loop) are described in the other documents under `docs/`.